# 03 特征选择 (Feature Selection)

从特征工程产出的 149 列中，筛选出有价值的特征。

---

## 本章工作评价（对比银牌参考方案）

### 完成情况
- 起始特征：149 列 → 筛选后：**110 列**
- 删除方式：去掉明显无用的特征（高相关冗余 + 极低方差）

### 做得好的地方
1. **高相关性过滤思路正确**：识别出 `_AVG/_MODE/_MEDI` 三列冗余模式，并结合业务理解选择保留 `_MODE`（偏态数据用众数更稳健）
2. **没有机械执行**：面对与 TARGET 相关性极低的特征，意识到线性相关性无法衡量非线性关系，选择跳过而非盲目删除，思路成熟
3. **保留了业务判断**：对缺失率高的列未急于删除，计划结合后续模型重要性再决策

### 与参考方案的差距
参考方案（银牌）采用**6种算法投票法**：
- Pearson 相关、Chi-2、RFE（逻辑回归）、L1正则化、随机森林、LightGBM
- 每个特征统计被几种算法选中，取票数最高的 100 列
- 这种方式更客观、更全面，能捕捉线性和非线性关系

本章采用的是**启发式粗筛**，计算成本低，逻辑清晰，适合作为第一步。

### 后续计划
- 在 04_baseline_model 中用 LightGBM 跑特征重要性，对剩余 110 列做精筛
- 对重要性为 0 的列和高缺失率列进行二次判断

In [31]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder


df= pd.read_csv("../data/processed/app_train_features.csv")
SK_ID_CURR = df["SK_ID_CURR"]

cat_cols = df.select_dtypes(exclude='number').columns

for col in cat_cols:
    le = LabelEncoder()
    df[col] = df[col].fillna('Missing')
    df[col] = le.fit_transform(df[col])
 
missing_rate = df.isnull().mean().sort_values(ascending=False)
low_variance = df.std().sort_values()
corr_matrix = df.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_pairs = [
    (col, row, upper.loc[row, col])
        for col in upper.columns
            for row in upper.index
                if upper.loc[row, col] > 0.95
]
high_corr = pd.DataFrame(high_corr_pairs, columns=['特征1', '特征2', '相关系数'])
high_corr.sort_values("相关系数",ascending=False)
cols_to_drop = [
    # _AVG 和 _MEDI 系列（保留 _MODE）
    'APARTMENTS_AVG', 'APARTMENTS_MEDI',
    'BASEMENTAREA_AVG', 'BASEMENTAREA_MEDI',
    'YEARS_BEGINEXPLUATATION_AVG', 'YEARS_BEGINEXPLUATATION_MEDI',
    'YEARS_BUILD_AVG', 'YEARS_BUILD_MEDI',
    'COMMONAREA_AVG', 'COMMONAREA_MEDI',
    'ELEVATORS_AVG', 'ELEVATORS_MEDI',
    'ENTRANCES_AVG', 'ENTRANCES_MEDI',
    'FLOORSMAX_AVG', 'FLOORSMAX_MEDI',
    'FLOORSMIN_AVG', 'FLOORSMIN_MEDI',
    'LANDAREA_AVG', 'LANDAREA_MEDI',
    'LIVINGAPARTMENTS_AVG', 'LIVINGAPARTMENTS_MEDI',
    'LIVINGAREA_AVG', 'LIVINGAREA_MEDI',
    'NONLIVINGAPARTMENTS_AVG', 'NONLIVINGAPARTMENTS_MEDI',
    'NONLIVINGAREA_AVG', 'NONLIVINGAREA_MEDI',
    # 其他高相关列
    'FLAG_EMP_PHONE',
    'AMT_GOODS_PRICE',
    'OBS_30_CNT_SOCIAL_CIRCLE',
    'REGION_RATING_CLIENT',
    'bureau_max_overdue_sum',
    'cc_dpd_mean',
    #极低标准差   <0.01
    'FLAG_MOBIL',
    'FLAG_DOCUMENT_12',
    'FLAG_DOCUMENT_10',
    'FLAG_DOCUMENT_2',
    'FLAG_DOCUMENT_4'
]
df = df.drop(columns= cols_to_drop)
df.to_csv("../data/processed/app_train_selected.csv", index=False)
print(f"已保存，shape: {df.shape}")


/home/ye/data-science/Home-Credit-Solution-like/.venv/lib/python3.10/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


已保存，shape: (307511, 110)
